In [15]:
from datetime import datetime,timedelta
import sys
sys.path.append('E:\wangzhilin\QuantumGalaxy')

import pandas as pd
import numpy as np
from QGI.neoapi import Neo4j
from QGI.feishu import *
from QGI.mysql import MYSQL
import numpy as np
import pandas as pd
import json
import scipy,math
from jira import JIRA
from server_new.utils import get_logger
logger=get_logger(__name__)
def get_neo():
    uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
    user = "QG_Editor"
    password = "editor"
    neo=Neo4j(uri,user,password)
    return neo
def get_backbone_index():
    neo=get_neo()
    backbones=neo.read_query("match (n)  where not  n.backbone contains ',' return distinct n.backbone ")
    return [backbone.values()[0] for backbone in backbones]

backbones=get_backbone_index()

['电力', '石油化工', '煤炭', '海运', '农牧业', '钢铁', '锂离子电池', '半导体', '石油天然气', '光伏', '钠离子电池']

In [27]:
a=2.4123123
a.__round__(2)

2.41

In [153]:
from random import random

import numpy as np

import matplotlib.pyplot as plt
import networkx as nx

import matplotlib as mpl
import json
import copy

import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL

import pandas as pd
import numpy as np
from QGI.neoapi import Neo4j
from QGI.feishu import *
from QGI.logger import get_logger
import os

In [154]:
logger=get_logger('backbone')

In [283]:
class QGDiGraph(nx.DiGraph):
    pass
class Backbone2JS():
    def __init__(self,name):
        self.name=name
        uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
        user = "QG_Editor"
        password = "editor"
        self.neo=Neo4j(uri,user,password)
        host='localhost'
        user='Local_Editor'
        password='QuantumGalaxy'
        
        self.mysql=MYSQL(host,user,password,'qgdbs')
        self.G=None
    def _close_neo(self):
        self.neo.close()
    def _restart_neo(self):
        uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
        user = "QG_Editor"
        password = "editor"
        self.neo=Neo4j(uri,user,password)
        return self.neo
    def _get_backbone_from_neo(self):
        '''first of everything'''
        backbone=self.name
        cypher='match (n) where n.backbone contains "%s" \
        return distinct n.name as name,labels(n) as labels,n.backbone as backbone,n.code as code \
        union all \
        match (n1) where n1.backbone contains "%s" \
        with n1 \
        match (n:Indicator)-[]-(n1) \
        return  distinct n.name as name,labels(n) as labels,n.backbone as backbone,n.code as code order by labels'%(backbone,backbone)
        nodes=self.neo.read_query(cypher)
        cypher1='match p=(n1)-[r]->(n2) where n1.backbone contains "%s" and n2.backbone contains "%s" return startNode(r).name as start,endNode(r).name as end,type(r) as rela \
        union all \
        match p=(n:Indicator)-[r]-(n1) where n1.backbone contains "%s"\
        return  distinct startNode(r).name as start,endNode(r).name as end,type(r) as rela'%(backbone,backbone,backbone)
        edges=self.neo.read_query(cypher1)
        self.nodes,self.edges=nodes,edges
        return (nodes,edges)
    
    def __get_backbone_from_mysql(self):
        '''不能用：找不到与骨干节点相连的indicator，要等骨干相关标的完善之后'''
        backbone=self.name
        sql='select name,bname,labels,code from backbone_nodes where bname="%s"'%backbone
        pass
    def _init_graph(self,nodes,edges):
        G=QGDiGraph()
        assert nx.is_directed_acyclic_graph(G)
        

        for node in nodes:
            G.add_node(node[0],**node[1:])
        for edge in edges:
            G.add_edge(edge[0],edge[1],relationship=edge[2])
        self.G=G
        return G
    def get_G(self) -> nx.Graph:
        n,e=self._get_backbone_from_neo()
        print(f'获取{self.name}骨干图，共有{len(n)}个节点，{len(e)}条边')
        return self._init_graph(n,e)


    def _fetch_indicator_data(self):
        date=self.mysql.read_query('select date from processed_data where code="000001.SZ" order by date desc limit 1')
        #std7d=pd.DataFrame(index=[d[0] for d in date])
        std1m=pd.DataFrame(index=[d[0] for d in date])
        #std3m=pd.DataFrame(index=[d[0] for d in date])
        nodes=self.nodes
        for node in nodes:
            code=node.get('code')
            labels=node.get('labels')
            #print(labels)
            if code and 'Indicator' in labels:
                #print(node)
                d=self.mysql.read_query('select date,stdchg7d,stdchg1m,stdchg3m from processed_data where code= "%s" order by date desc limit 30'%code)
                #df7d=pd.DataFrame(index=[row[0]for row in d],columns=[code],data=[row[1] for row in d])
                df1m=pd.DataFrame(index=[row[0]for row in d],columns=[code],data=[row[2] for row in d])
                #df3m=pd.DataFrame(index=[row[0]for row in d],columns=[code],data=[row[3] for row in d])
                #std7d=pd.concat([std7d,df7d],axis=1)
                std1m=pd.concat([std1m,df1m],axis=1)
                #std3m=pd.concat([std3m,df3m],axis=1)
        #std7d=std7d.sort_index().fillna(method='bfill').dropna()
                #std7d.to_csv(r'E:\wangzhilin\QuantumGalaxy\backbone\raw_data\std7d.csv')
        std1m=std1m.sort_index().fillna(method='bfill').dropna()
                #std1m.to_csv(r'E:\wangzhilin\QuantumGalaxy\backbone\raw_data\std1m.csv')
        #std3m=std3m.sort_index().fillna(method='bfill').dropna()
                #std3m.to_csv(r'E:\wangzhilin\QuantumGalaxy\backbone\raw_data\std3m.csv')
        #self.std1m=std1m
        #print(std1m)
        return std1m

    def _gen_js_1day(self,one_day_data:pd.Series):
        '''从所有indicator边传导到对应的产品/工艺的供需差（数据用stdchg1m中某一天），在各节点加权平均（权重暂为1，考虑用市值/量利等），得到多个在节点上的初始激励'''
        if not self.G:
            self.get_G()
        G=copy.deepcopy(self.G)
        
        #print(std1m)
        dic={}
        
        for name,attr in self.G.nodes(True):
            if attr['code']:
                dic[attr['code']]=name

        '''
        for code,data in one_day_data.iteritems():
                node=dic[code]
                self.G.nodes[node]['std']=data
                #print(self.G.nodes[node])
                
        
        for n,preds in self.G.pred.items():
            if preds:
                values=[]
                for pred in preds:
                    std=self.G.nodes.data('std')[pred]
                    
                    if std:
                        values.append(std)
                if len(values)!=0:
                    self.G.nodes[n]['i']=np.average(values)
        [(n,d) for (n,d) in self.G.nodes.data('i') if d]
        '''       

        #每天为一组数据，放到nodes/edges里
        #print(date)
        datas=one_day_data.values
        max=np.max(datas)
        min=np.min(datas)
        column=one_day_data.index
        for code,data in one_day_data.iteritems():
            node=dic[code]
            G.nodes[node]['std']=data

            #nor=normalize(data,datas)
            #colorstr=get_cmap_color_str(nor,regrcmp)
            
            #print(colorstr)
        for u,v,attr in G.edges(data=True):
            if 'Indicator' in  G.nodes[u]['labels'] :
                if 'measure' in attr['relationship'] or 'produce' in attr['relationship']:
                    attr['velocity']=G.nodes[u]['std']
                    #print(attr['velocity'],datas)
                    nor=normalize(attr['velocity'],max,min)
                    colorstr=get_cmap_color_str(nor,regrcmp)
                    attr['color']=colorstr
                    
            else:
                attr['velocity']=0
            #a=Backbone_handler(G)


            node_json=[]
            stock=list(nx.get_node_attributes(G,'std').values())
            for n in copy.deepcopy(G.nodes(data=True)):
                name,attr=n
                attr['name']=name
                attr['labels']=attr.pop('labels')
                #attr['color']=get_cmap_color_str(normalize(attr['stock'],stock),regrcmp)
                node_json.append(attr)
                
            edge_json=[]
            velocity=list(nx.get_edge_attributes(G,'velocity').values())

            for e in copy.deepcopy(G.edges(data=True)):
                s,end,attr=e
                attr['source']=s
                attr['target']=end
                #print(attr)
                #attr['color']=get_cmap_color_str(normalize(attr['velocity'],velocity),regrcmp)
                #attr['color']=
                #
                edge_json.append(attr)
            #print(edge_json)
        #self.G=G
        return node_json,edge_json
    def gen_js(self,):
        std1m=self._fetch_indicator_data()
        jsdata={}
        jsdata['categories']=[{'name':"Product"},{'name':"Technique"}, {'name':"Company"}, {'name':"Indicator"}]
        jsdata['dates']=std1m.index.astype(str).tolist()
        jsdata['nodes']=[]
        jsdata['links']=[]
        
        
        for date,r in std1m.iterrows():
            #print(date,r['600584.SH'])
            n,e=self._gen_js_1day(r)
            jsdata['nodes'].append(n)
            jsdata['links'].append(e)
        return jsdata


In [284]:
def normalize(data,max,min):

    return (data-min)/(max-min)
def get_cmap_color_str(data,cmap):
    '''从正则后的颜色值，获取指定cmap中的颜色，并转化为echarts需要的“rgba(255,255,255,1)”形式'''
    c=cmap(data)
    colors=(c[0]*255,c[1]*255,c[2]*255,1)
    return(f'rgba{colors}')
cdict = {'red':   [[0.0,  91/255, 91/255],
                   [0.5,  200/255, 200/255],
                   [1.0,  255/255, 255/255]],
         'green': [[0.0,  218/255, 218/255],
                   [0.5, 200/255, 200/255],
                   [1, 94/255, 94/255]],

         'blue':  [[0.0,  146/255, 146/255],
                   [0.5,  121.5/255, 121.5/255],
                   [1.0,  97/255, 97/255]]}

cdict = {'red':   [[0.0,  121/255, 121/255],
                   
                   [1.0,  241/255, 241/255]],
         'green': [[0.0,  240/255, 240/255],
                   
                   [1, 70/255, 70/255]],

         'blue':  [[0.0,  120/255, 120/255],
                   
                   [1.0,  103/255, 103/255]]}
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.cm import cool
regrcmp = LinearSegmentedColormap('testCmap', segmentdata=cdict, N=256)
rgba = regrcmp(np.linspace(0, 1, 256))

In [285]:
b=Backbone2JS('半导体')
b._get_backbone_from_neo()
js=b.gen_js()



获取半导体骨干图，共有66个节点，74条边


In [274]:
jsdata=js

In [275]:
js=json.dumps(jsdata,ensure_ascii=False)
js='graph='+js

file=open(r'E:\wangzhilin\QuantumGalaxy\server_new\blueprints\backbone\static\full_data.js','w',encoding='UTF-8')
file.write(js)
file.close()

In [276]:
G.nodes('std')

NodeDataView({'石墨': None, 'IGBT芯片': None, '芯片': None, '高纯石英砂': None, '功率半导体': None, '薄膜沉积': None, '半导体薄膜沉积设备': None, '芯片代工': None, '半导体传感器': None, '晶圆抛光': None, '离子注入': None, '晶圆刻蚀': None, '光刻': None, '半导体分立器件': None, '光刻机': None, '光刻胶': None, '晶圆': None, '半导体硅片衬底': None, '驱动IC': None, '玻璃': None, '电源管理模块': None, '光模块': None, '逻辑芯片': None, '发光二极管': None, '半导体碳化硅衬底': None, '光掩膜基板': None, '功率芯片': None, '芯片封装测试': None, 'EDA工具软件': None, '物理气相传输法': None, 'IGBT模块': None, '功率模块': None, '光电子器件': None, 'CMP抛光材料': None, '半导体': None, '模拟芯片': None, '中芯国际': None, '东微半导': None, '伊戈尔': None, '台积电': None, '聚辰股份': None, '国芯科技': None, '华天科技': None, '信义玻璃': None, '天岳先进': None, '美光科技': None, '豪微': None, '菲利华': None, '利扬芯片': None, '华海清科': None, '气派科技': None, '长电科技': None, '通富微电': None, '金晶科技': None, '华大九天': None, '新思科技': None, '概伦电子': None, '时代电气': None, '华润微': None, '晶方科技': None, '露笑科技': None, '光迅科技': None, '博通': None, '大港股份': None, '广和通': None}, data='std')

In [216]:
G.edges(data=True)

OutEdgeDataView([('石墨', '物理气相传输法', {'relationship': 'compose'}), ('IGBT芯片', 'IGBT模块', {'relationship': 'compose'}), ('芯片', '光模块', {'relationship': 'compose'}), ('芯片', '模拟芯片', {'relationship': 'parent'}), ('芯片', '逻辑芯片', {'relationship': 'parent'}), ('芯片', '功率芯片', {'relationship': 'parent'}), ('高纯石英砂', '玻璃', {'relationship': 'compose'}), ('高纯石英砂', '物理气相传输法', {'relationship': 'compose'}), ('功率半导体', '功率芯片', {'relationship': 'compose'}), ('半导体薄膜沉积设备', '薄膜沉积', {'relationship': 'compose'}), ('芯片代工', '芯片', {'relationship': 'compose'}), ('芯片代工', '芯片代工', {'relationship': 'compose'}), ('芯片代工', '晶圆刻蚀', {'relationship': 'parent'}), ('芯片代工', '光刻', {'relationship': 'parent'}), ('芯片代工', '薄膜沉积', {'relationship': 'parent'}), ('芯片代工', '离子注入', {'relationship': 'parent'}), ('芯片代工', '晶圆抛光', {'relationship': 'parent'}), ('半导体分立器件', '半导体传感器', {'relationship': 'parent'}), ('半导体分立器件', '功率半导体', {'relationship': 'parent'}), ('半导体分立器件', '光电子器件', {'relationship': 'parent'}), ('光刻机', '光刻', {'relationship': 'compose'

In [214]:
r=std1m.iloc[1]

In [215]:
for a,b in r.iteritems():
    print("a"+a)
    print(b)
    

a688981.SH
1.121
a688261.SH
0.420686
a002922.SZ
1.19252
aTSM.N
-1.06541
a688123.SH
1.40326
a688262.SH
0.750584
a002185.SZ
0.803629
a0868.HK
-0.185472
a688234.SH
0.372062
aMU.O
0.220091
aNA.O
-0.784701
a300395.SZ
0.270322
a688135.SH
0.047627
a688120.SH
-0.704875
a688216.SH
0.549415
a600584.SH
0.970472
a002156.SZ
0.966529
a600586.SH
-0.269575
a301269.SZ
0.680348
aSNPS.O
-0.759767
a688206.SH
0.794129
a688187.SH
0.758659
a688396.SH
0.728479
a603005.SH
0.682999
a002617.SZ
0.598462
a002281.SZ
1.13198
aAVGO.O
0.202218
a002077.SZ
1.25205
a300638.SZ
0.56042


In [201]:
G.nodes('std')

NodeDataView({'石墨': None, 'IGBT芯片': None, '芯片': None, '高纯石英砂': None, '功率半导体': None, '薄膜沉积': None, '半导体薄膜沉积设备': None, '芯片代工': None, '半导体传感器': None, '晶圆抛光': None, '离子注入': None, '晶圆刻蚀': None, '光刻': None, '半导体分立器件': None, '光刻机': None, '光刻胶': None, '晶圆': None, '半导体硅片衬底': None, '驱动IC': None, '玻璃': None, '电源管理模块': None, '光模块': None, '逻辑芯片': None, '发光二极管': None, '半导体碳化硅衬底': None, '光掩膜基板': None, '功率芯片': None, '芯片封装测试': None, 'EDA工具软件': None, '物理气相传输法': None, 'IGBT模块': None, '功率模块': None, '光电子器件': None, 'CMP抛光材料': None, '半导体': None, '模拟芯片': None, '中芯国际': 2022-11-03    1.2032
Name: 688981.SH, dtype: float64, '东微半导': 2022-11-03    0.671216
Name: 688261.SH, dtype: float64, '伊戈尔': 2022-11-03    1.36941
Name: 002922.SZ, dtype: float64, '台积电': 2022-11-03   NaN
Name: TSM.N, dtype: float64, '聚辰股份': 2022-11-03    1.87009
Name: 688123.SH, dtype: float64, '国芯科技': 2022-11-03    0.838123
Name: 688262.SH, dtype: float64, '华天科技': 2022-11-03    0.892469
Name: 002185.SZ, dtype: float64, '信义玻璃': 2022-11-03   -0.4

In [121]:

b.__dict__

{'name': '半导体',
 'neo': <QGI.neoapi.Neo4j at 0x2268f922c10>,
 'mysql': <QGI.mysql.MYSQL at 0x2268f9225b0>,
 'G': <__main__.QGDiGraph at 0x2268fa0d970>}

In [106]:
name='半导体'
G=Backbone2JS(name).get_G()
G

获取半导体骨干图，共有66个节点，74条边


In [96]:
G.number_of_edges()

0

In [85]:

G=b.init_graph()
for a in nx.ancestors(G,'氯化钠'):
    print(G.nodes[a]['labels'])
    

['Product']
['Company', 'Indicator']


In [88]:
G.nodes(data=True)

NodeDataView({'聚乙烯': {'labels': ['Product'], 'backbone': '锂离子电池,钠离子电池,石油化工', 'code': None}, '聚丙烯': {'labels': ['Product'], 'backbone': '锂离子电池,钠离子电池,石油化工', 'code': None}, '隔膜添加剂': {'labels': ['Product'], 'backbone': '锂离子电池,钠离子电池', 'code': None}, '隔膜': {'labels': ['Product'], 'backbone': '锂离子电池,钠离子电池,石油化工', 'code': None}, '电解液添加剂': {'labels': ['Product'], 'backbone': '锂离子电池,钠离子电池', 'code': None}, '电解液溶剂': {'labels': ['Product'], 'backbone': '锂离子电池,钠离子电池', 'code': None}, '石油焦': {'labels': ['Product'], 'backbone': '钠离子电池', 'code': None}, '导电剂': {'labels': ['Product'], 'backbone': '锂离子电池,钠离子电池', 'code': None}, '铝箔': {'labels': ['Product'], 'backbone': '锂离子电池,钠离子电池', 'code': None}, '电池后段设备': {'labels': ['Product'], 'backbone': '锂离子电池,钠离子电池', 'code': None}, '电池中段设备': {'labels': ['Product'], 'backbone': '锂离子电池,钠离子电池', 'code': None}, '电池前段设备': {'labels': ['Product'], 'backbone': '锂离子电池,钠离子电池', 'code': None}, '电池生产设备': {'labels': ['Product'], 'backbone': '锂离子电池,钠离子电池', 'code': None}, '沥青': {'lab

In [69]:
df=b.fetch_indicator_data()
df.sort_index().fillna(method='bfill').dropna()

,002812.SZ,300450.SZ,603659.SH,300750.SZ,300586.SZ,688819.SH,002866.SZ,003027.SZ,002125.SZ,300140.SZ,...,000807.SZ,601677.SH,000933.SZ,3903.HK,301123.SZ,301180.SZ,002866.SZ,003027.SZ,002125.SZ,BU.SHF
2022-09-22,-1.331950,-1.483550,-1.087410,-1.945270,-0.677159,-1.371030,0.752041,1.460180,-0.783763,-1.111170,...,-0.511191,-1.338060,0.446045,-1.104000,-1.734610,-1.396680,0.752041,1.460180,-0.783763,-0.412454
2022-09-23,-1.031590,-1.455670,-0.845730,-1.508890,-0.696088,-1.317390,0.940999,1.965010,-0.619567,-0.882906,...,-0.643533,-1.229640,0.261218,-0.745992,-1.530000,-1.554860,0.940999,1.965010,-0.619567,-0.364348
2022-09-26,-0.585295,-1.009010,-0.564368,-1.098790,-0.439041,-1.144300,1.157220,2.302350,-0.565648,-0.521565,...,-1.131980,-1.379490,-0.393297,-1.350960,-0.919057,-1.168590,1.157220,2.302350,-0.565648,-0.388960
2022-09-27,-0.558029,-0.887398,-0.611541,-1.086170,-0.449968,-1.061090,1.185470,2.674000,-0.556478,-0.405893,...,-1.071280,-1.412120,-0.538168,-1.350960,-0.829038,-1.041080,1.185470,2.674000,-0.556478,-0.343427
2022-09-28,-0.728889,-1.230900,-1.114110,-1.211860,-0.872570,-1.104450,0.851597,2.081410,-0.768223,-0.729777,...,-1.281420,-1.598440,-0.860866,-1.350960,-1.004570,-1.188910,0.851597,2.081410,-0.768223,-0.378387
2022-09-29,-0.438827,-1.145170,-0.860230,-1.076220,-0.681818,-0.777281,0.925960,2.249040,-0.458388,-0.626428,...,-0.982150,-1.357960,-0.322270,-1.350960,-0.970063,-1.135300,0.925960,2.249040,-0.458388,-0.138429
2022-09-30,-0.809176,-1.131210,-1.031210,-1.236450,-0.464397,-0.754771,1.023570,1.755440,-0.290485,-0.499997,...,-0.579787,-1.129230,0.217300,-1.979030,-0.876754,-1.066130,1.023570,1.755440,-0.290485,0.197633
2022-10-03,-0.700942,-0.647235,-0.874342,-1.018590,-0.484112,-0.580352,0.500994,1.680110,-0.351175,-0.451470,...,-0.556719,-1.509860,0.085901,-1.979030,-0.878125,-1.070540,0.500994,1.680110,-0.351175,0.264074
2022-10-04,-0.675873,-0.764666,-0.888946,-1.056310,-0.458853,-0.726858,0.245108,1.674720,-0.338211,-0.493648,...,-0.441442,-1.266530,-0.003128,-1.756460,-0.788025,-1.009220,0.245108,1.674720,-0.338211,0.219418
2022-10-05,-0.675873,-0.764666,-0.888946,-1.056310,-0.458853,-0.726858,0.245108,1.674720,-0.338211,-0.493648,...,-0.441442,-1.266530,-0.003128,-1.756460,-0.788025,-1.009220,0.245108,1.674720,-0.338211,0.219418


In [72]:
dic=(**node[1:])
dic

'锂离子电池,钠离子电池'

In [22]:
node=nodes[0]


In [24]:
node.get('name')

In [3]:
backbone='电力'
uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
user = "QG_Editor"
password = "editor"
neo=Neo4j(uri,user,password)

cypher='match (n) where n.backbone contains "%s" \
return distinct n.name,labels(n),n.backbone,n.code \
union all \
match (n1) where n1.backbone contains "%s" \
with n1 \
match (n:Indicator)-[]-(n1) \
return  distinct n.name,labels(n),n.backbone,n.code order by labels(n)'%(backbone,backbone)
nodes=neo.read_query(cypher)
cypher='match p=(n1)-[r]->(n2) where n1.backbone contains "%s" and n2.backbone contains "%s" return startNode(r).name,endNode(r).name,type(r) \
union all \
match p=(n:Indicator)-[r]-(n1) where n1.backbone contains "%s"\
return  distinct startNode(r).name,endNode(r).name,type(r)'%(backbone,backbone,backbone)
edges=neo.read_query(cypher)

In [19]:
nodes

[<Record n.name='风电厂' labels(n)=['Product'] n.backbone='电力' n.code=None>,
 <Record n.name='核电站' labels(n)=['Product'] n.backbone='电力' n.code=None>,
 <Record n.name='继电器' labels(n)=['Product'] n.backbone='电力' n.code=None>,
 <Record n.name='光伏系统' labels(n)=['Product'] n.backbone='电力,光伏' n.code='None'>,
 <Record n.name='发电厂' labels(n)=['Product'] n.backbone='电力' n.code=None>,
 <Record n.name='储能锂电池' labels(n)=['Product'] n.backbone='锂离子电池,电力' n.code=None>,
 <Record n.name='火力发电' labels(n)=['Technique'] n.backbone='电力,煤炭' n.code=None>,
 <Record n.name='动力煤' labels(n)=['Product'] n.backbone='电力,煤炭' n.code=None>,
 <Record n.name='电力' labels(n)=['Product'] n.backbone='电力,光伏' n.code=None>,
 <Record n.name='天然气' labels(n)=['Product'] n.backbone='石油天然气,电力' n.code=None>,
 <Record n.name='火电厂' labels(n)=['Product'] n.backbone='电力' n.code=None>,
 <Record n.name='逆变器' labels(n)=['Product'] n.backbone='电力' n.code=None>,
 <Record n.name='变压器' labels(n)=['Product'] n.backbone='电力' n.code=None>,
 <Recor

In [18]:
edges[0]

<Record startNode(r).name='风电厂' endNode(r).name='风储一体化' type(r)='compose'>

In [ ]:
def work():
    mysql=MYSQL()